# F1 · ETL Capa 1 — Partidos, equipos y rankingsEjecuta la **Capa 1** con *checkpoint/resume*:1. Resultados recientes (`/v2/match?q=results`, limitado a 100 páginas por la API).2. Histórico completo vía eventos (`/v2/events` + `/v2/events/matches`).3. Rankings regionales (`/v2/rankings`).4. Datos de equipo (perfil, stats por mapa, historial, roster).Escribe crudos en `data/raw/` y tablas consolidadas (Parquet) en `data/interim/`.Puedes interrumpir y re-ejecutar: solo descarga lo que falta.

## 1. Montar Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Rutas

In [ ]:
import sys
from pathlib import Path

# Ajusta esta ruta a donde tengas el repo dentro de tu Drive.
DRIVE_BASE = "/content/drive/MyDrive/Analisis de datos/Proyecto final"
# Si prefieres el disco local de Colab (más rápido, efímero), descomenta:
# DRIVE_BASE = "/content/Proyecto final"

REPO_DIR = Path(DRIVE_BASE)
DATA_DIR = REPO_DIR / "data"
SRC_DIR = REPO_DIR / "src"
API_DIR = REPO_DIR / "vlrggapi"
BASE_URL = "http://127.0.0.1:3001"

for d in (REPO_DIR, DATA_DIR, SRC_DIR):
    d.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(SRC_DIR))
print("REPO_DIR:", REPO_DIR)
print("DATA_DIR:", DATA_DIR)


## 3. Dependencias + clonar vlrggapi

In [ ]:
# Dependencias del proyecto (pandas, pyarrow, requests, tqdm, ...).
!pip install -q -r "{REPO_DIR}/requirements.txt"
# Dependencias de vlrggapi (uv no está en Colab; usamos pip directamente).
!pip install -q fastapi "uvicorn[standard]" httpx selectolax cachetools

# Clona vlrggapi (la instancia hosteada está caída). Si ya existe, no pasa nada.
!git clone https://github.com/axsddlr/vlrggapi.git "{API_DIR}" 2>/dev/null || echo "vlrggapi ya clonado"


## 4. Servidor + cliente

In [ ]:
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(name)s %(levelname)s %(message)s")

from scraper import VLRClient, start_server, wait_for_api

# Arranca el servidor si no responde (idempotente entre notebooks/runtime).
if not wait_for_api(BASE_URL, timeout=5):
    proc = start_server(API_DIR, port=3001, log_file=DATA_DIR / "_vlrggapi.log")
    wait_for_api(BASE_URL, timeout=240)

client = VLRClient(base_url=BASE_URL, data_dir=DATA_DIR)
print("API lista:", client.version())


## 5. Ejecutar Capa 1

In [ ]:
from etl import run_layer1

tables = run_layer1(
    client,
    max_results_pages=100,    # q=results limitado a 100 páginas por la API
    pages_per_request=5,      # ventana por request (máx 20); checkpoint por ventana
    max_events_pages=100,     # páginas de eventos completados
    max_team_match_pages=3,   # páginas de historial por equipo (para la forma)
    rankings=True,
    team_data=True,
)


## 6. Sanity check

In [ ]:
for name, df in tables.items():
    print(f"{name:16s} {df.shape}")

m = tables["matches"]
print(m.head())
print("\nNulos en winner:", m["winner"].isna().sum(), "/", len(m))
print("Distribución de fuente:", m["source"].value_counts(dropna=False).to_dict())
